# F15 — Preflight de shards DL en Vast.ai

Este cuaderno prepara una instancia **Vast.ai PyTorch** para F16. No descarga
ZZU crudo, no genera shards y no usa Google Drive, Tailscale ni credenciales.

Antes de abrirlo, carga desde tu equipo únicamente estos archivos al clon:

```text
results/dl/shards/shard_000.npy ... shard_006.npy
results/dl/manifiesto.csv
results/dl/splits.csv
results/dl/normalizacion.json
results/dl/verificacion_shards.txt
```

Destino esperado: `/workspace/ecg/results/dl/`. Los shards ocupan ~6,6 GB.
No cargar `data/zzu_pecg/`: F16 no lo necesita. La instancia debe tener una GPU
compatible con CUDA 12.8+ para GPUs RTX 50-series y espacio libre suficiente.

## 1. Conexión y carga desde el equipo local

Vast ya publica SSH directo. No inicies `sshd`, no instales Tailscale y no
copies claves privadas al servidor. Desde el equipo local, usa `scp` o `rsync`
con el host, puerto e identidad que entrega Vast. Ejemplo ilustrativo:

```bash
rsync -avP -e 'ssh -i ~/.ssh/keys/vast-ai/id_ed25519 -p <PUERTO>' \
  results/dl/ root@<IP>:/workspace/ecg/results/dl/
```

El comando copia datos; no subas secretos, el dataset crudo ni directorios de
entorno. Consulta `docs/runbooks/vastai_dl.md` para el flujo completo.

In [ ]:
from pathlib import Path
import os

REPO = Path(os.environ.get('ECG_REPO', '/workspace/ecg')).resolve()
DL = REPO / 'results' / 'dl'
assert (REPO / 'scripts' / 'entrenar_baseline_dl.py').exists(), (
    f'No encuentro el repo en {REPO}. Clónalo o define ECG_REPO.')
print('Repo:', REPO)
print('DL:', DL)

In [ ]:
import subprocess
import sys

# La plantilla PyTorch de Vast trae torch; estas son dependencias ligeras del loader.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'numpy', 'pandas'],
               check=True)

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), 'No hay GPU CUDA disponible en esta instancia.'
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['df', '-h', str(REPO)], check=True)

In [ ]:
import json

requeridos = [
    DL / 'manifiesto.csv', DL / 'splits.csv',
    DL / 'normalizacion.json', DL / 'verificacion_shards.txt',
]
faltan = [str(p) for p in requeridos if not p.is_file()]
shards = sorted((DL / 'shards').glob('shard_*.npy'))
assert not faltan, f'Faltan artefactos ligeros: {faltan}'
assert len(shards) == 7, f'Esperaba 7 shards, encontré {len(shards)}'

man = pd.read_csv(DL / 'manifiesto.csv')
formas = [np.load(p, mmap_mode='r').shape for p in shards]
assert all(s[1:] == (12, 5000) for s in formas), formas
assert sum(s[0] for s in formas) == len(man) == 58178
norm = json.loads((DL / 'normalizacion.json').read_text())
assert len(norm['media_mv']) == len(norm['desv_mv']) == 12
print('F15 PRECHECK VERDE')
print('Ventanas:', len(man), '| formas:', formas)
print('Clases:', man.clase.value_counts().to_dict())
print((DL / 'verificacion_shards.txt').read_text())

## Siguiente paso

Si el preflight termina en verde, abre `06_dl_baseline_colab.ipynb` y ejecuta
primero el humo. El F16 completo guarda sus salidas bajo `runs/`; descárgalas
al equipo local antes de destruir o reciclar la instancia.